In [14]:
# instacart feature engineering

import pandas as pd
import os

# step 1. function to load master file
def load_master():
    base = "D:\\instacart_market_analysis\\"
    # to make master file size small 
    orders       = pd.read_csv(base + "cleaned_orders.csv",
                               dtype={
                        "order_id":      "int32",
                        "user_id":       "int32",
                        "order_number":  "int16",
                        "order_dow":     "int8",
                        "order_hour_of_day": "int8",
                        "days_since_prior_order": "float32"
                })

    products     = pd.read_csv(base + "cleaned_products.csv")
    aisles       = pd.read_csv(base + "cleaned_aisles.csv")
    departments  = pd.read_csv(base + "cleaned_departments.csv")
    prior        = pd.read_csv(base + "cleaned_order_products__prior.csv",
                     dtype={
                        "order_id":          "int32",
                        "product_id":        "int32",
                        "add_to_cart_order": "int16",
                        "reordered":         "int8"
                })  
               

    master = prior.merge(orders,      on="order_id",      how="left")
    master = master.merge(products,   on="product_id",    how="left")
    master = master.merge(aisles,     on="aisle_id",      how="left")
    master = master.merge(departments,on="department_id", how="left")

    return master

master = load_master()

# step 2. make functions to add column in master table
# 1. Define the function to assign time of day
def assign_time(hour):
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'



# 2. Define the function to assign day type
def assign_day(day):
    if day in [1, 2, 3, 4, 5]:  # Monday to Friday
        return 'Weekday'
    else:  # Saturday and Sunday
        return 'Weekend'
    

#3 Define the function to assign order frequency
def assign_order_frequency(days):
    if days == -1:
        return 'First order'
    elif days <= 7:
        return 'Weekly'
    elif days <= 15:
        return 'Bi Weekly'
    else:
        return 'Monthly'

#4 Define the function to assign order stage
def order_stage(order_number):
    if order_number <=2: 
     return 'New Customer'
    elif order_number <= 8:
        return 'Growing' 
    else:
        return 'Loyal'
    
# step 3. Add columns to master table 
master['time_of_order'] = master['order_hour_of_day'].apply(assign_time)   
master['day_type'] = master['order_dow'].apply(assign_day)
master['order_frequency'] = master['days_since_prior_order'].apply(assign_order_frequency)
master['order_stage'] = master['order_number'].apply(order_stage)
## step 4. add 1 more column to master table


basket = master.groupby("order_id").agg(
    basket_size          = ("product_id", "count"),
    order_reorder_count  = ("reordered",  "sum"),
).reset_index()

basket["order_reorder_rate"] = (
    basket["order_reorder_count"] / basket["basket_size"]
).round(2)

master = master.merge(basket, on="order_id", how="left")

# step 5. save master table in parquet file to small its size 

master.to_parquet("D:\\instacart_market_analysis\\master_table.parquet", index=False)



In [15]:
# check master table 
import pandas as pd
df = pd.read_parquet("D:\\instacart_market_analysis\\master_table.parquet")
df.head(50)

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,...,department_id,aisle,department,time_of_order,day_type,order_frequency,order_stage,basket_size,order_reorder_count,order_reorder_rate
0,2,33120,1,1,202279,prior,3,5,9,8.0,...,16,Eggs,Dairy Eggs,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
1,2,28985,2,1,202279,prior,3,5,9,8.0,...,4,Fresh Vegetables,Produce,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
2,2,9327,3,0,202279,prior,3,5,9,8.0,...,13,Spices Seasonings,Pantry,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
3,2,45918,4,1,202279,prior,3,5,9,8.0,...,13,Oils Vinegars,Pantry,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
4,2,30035,5,0,202279,prior,3,5,9,8.0,...,13,Baking Ingredients,Pantry,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
5,2,17794,6,1,202279,prior,3,5,9,8.0,...,4,Fresh Vegetables,Produce,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
6,2,40141,7,1,202279,prior,3,5,9,8.0,...,13,Doughs Gelatins Bake Mixes,Pantry,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
7,2,1819,8,1,202279,prior,3,5,9,8.0,...,13,Spreads,Pantry,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
8,2,43668,9,0,202279,prior,3,5,9,8.0,...,4,Packaged Vegetables Fruits,Produce,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
9,3,33754,1,1,205970,prior,16,5,17,12.0,...,16,Yogurt,Dairy Eggs,Evening,Weekday,Bi Weekly,Loyal,8,8,1.00
